# Dutch Traffic Speed Data Exploration

This notebook explores traffic speed and flow data from the Netherlands National Data Warehouse (NDW) in DATEX II XML format.

## Data Description
The data contains:
- Traffic flow measurements (vehicles per hour)
- Traffic speed measurements (km/h)
- Multiple measurement sites across the Netherlands
- Timestamp: October 23, 2025

## 1. Setup and Import Libraries

In [ ]:
import gzip
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully!")

## 2. Load and Parse XML Data

In [ ]:
# File path
file_path = r'C:\Users\nicol\Documents\TrafficOntology_Project\TrafficOntology\data_raw\other data\trafficspeed.xml.gz'

# Read and decompress the file
print("Loading compressed XML file...")
with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    xml_content = f.read()

print(f"File loaded. Size: {len(xml_content):,} characters")

In [ ]:
# Parse XML
print("Parsing XML...")
root = ET.fromstring(xml_content)

# Define namespaces
namespaces = {
    'SOAP': 'http://schemas.xmlsoap.org/soap/envelope/',
    'd2': 'http://datex2.eu/schema/2/2_0',
    'xsi': 'http://www.w3.org/2001/XMLSchema-instance'
}

print("XML parsed successfully!")

## 3. Extract Measurement Site Locations

In [ ]:
# Function to extract site locations from measurement site table
def extract_site_locations(root, namespaces):
    """
    Extract location information for each measurement site.
    DATEX II format includes measurement site table with coordinates.
    """
    site_locations = {}
    
    # Find measurement site table
    for measurement_site in root.findall('.//d2:measurementSiteRecord', namespaces):
        # Get site ID
        site_id = measurement_site.get('id', 'Unknown')
        
        # Initialize location data
        location_data = {
            'site_id': site_id,
            'latitude': None,
            'longitude': None,
            'road_name': None,
            'direction': None
        }
        
        # Extract point coordinates if available
        point_coords = measurement_site.find('.//d2:pointByCoordinates', namespaces)
        if point_coords is not None:
            lat = point_coords.find('.//d2:latitude', namespaces)
            lon = point_coords.find('.//d2:longitude', namespaces)
            if lat is not None and lon is not None:
                location_data['latitude'] = float(lat.text)
                location_data['longitude'] = float(lon.text)
        
        # Alternative: Extract from Point element
        if location_data['latitude'] is None:
            point = measurement_site.find('.//d2:locationForDisplay/d2:latitude', namespaces)
            if point is not None:
                location_data['latitude'] = float(point.text)
            point = measurement_site.find('.//d2:locationForDisplay/d2:longitude', namespaces)
            if point is not None:
                location_data['longitude'] = float(point.text)
        
        # Extract road information
        road = measurement_site.find('.//d2:roadNumber', namespaces)
        if road is not None:
            location_data['road_name'] = road.text
        
        # Extract direction
        direction = measurement_site.find('.//d2:directionBound', namespaces)
        if direction is not None:
            location_data['direction'] = direction.text
        
        site_locations[site_id] = location_data
    
    return site_locations

# Extract site locations
print("Extracting measurement site locations...")
site_locations = extract_site_locations(root, namespaces)
print(f"Extracted location data for {len(site_locations):,} measurement sites")

# Show sample of sites with coordinates
sites_with_coords = {k: v for k, v in site_locations.items() if v['latitude'] is not None}
print(f"Sites with coordinates: {len(sites_with_coords):,}")

if len(sites_with_coords) > 0:
    print("\nSample site locations:")
    for i, (site_id, loc) in enumerate(list(sites_with_coords.items())[:3]):
        print(f"  {site_id}: ({loc['latitude']:.6f}, {loc['longitude']:.6f})")
else:
    print("\n⚠️  No coordinate data found in measurement site table.")
    print("    Coordinates may need to be joined from external source.")

## 4. Extract Traffic Measurements

In [ ]:
# Function to extract measurements from XML
def extract_measurements(root, namespaces):
    measurements = []
    
    # Find all siteMeasurements elements
    for site_measurement in root.findall('.//d2:siteMeasurements', namespaces):
        # Get site reference
        site_ref = site_measurement.find('.//d2:measurementSiteReference', namespaces)
        site_id = site_ref.get('id') if site_ref is not None else 'Unknown'
        
        # Get measurement time
        time_elem = site_measurement.find('.//d2:measurementTimeDefault', namespaces)
        measurement_time = time_elem.text if time_elem is not None else None
        
        # Process each measured value
        for measured_value in site_measurement.findall('.//d2:measuredValue', namespaces):
            index = measured_value.get('index')
            
            # Check for traffic flow data
            flow_elem = measured_value.find('.//d2:vehicleFlowRate', namespaces)
            if flow_elem is not None:
                measurements.append({
                    'site_id': site_id,
                    'timestamp': measurement_time,
                    'measurement_index': index,
                    'measurement_type': 'flow',
                    'value': float(flow_elem.text),
                    'unit': 'vehicles/hour'
                })
            
            # Check for traffic speed data
            speed_elem = measured_value.find('.//d2:speed', namespaces)
            if speed_elem is not None:
                speed_value = float(speed_elem.text)
                
                # Get additional speed metadata
                avg_speed = measured_value.find('.//d2:averageVehicleSpeed', namespaces)
                num_values = avg_speed.get('numberOfInputValuesUsed') if avg_speed is not None else None
                std_dev = avg_speed.get('standardDeviation') if avg_speed is not None else None
                
                measurements.append({
                    'site_id': site_id,
                    'timestamp': measurement_time,
                    'measurement_index': index,
                    'measurement_type': 'speed',
                    'value': speed_value,
                    'unit': 'km/h',
                    'num_values_used': int(num_values) if num_values else None,
                    'std_deviation': float(std_dev) if std_dev else None
                })
    
    return measurements

# Extract all measurements
print("Extracting measurements...")
measurements = extract_measurements(root, namespaces)
print(f"Extracted {len(measurements):,} measurements")

## 5. Create DataFrames for Analysis

In [ ]:
# Create main DataFrame
df = pd.DataFrame(measurements)

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Create location DataFrame and merge coordinates
df_locations = pd.DataFrame(site_locations.values())
if len(df_locations) > 0:
    df = df.merge(df_locations[['site_id', 'latitude', 'longitude', 'road_name', 'direction']], 
                  on='site_id', how='left')
    print(f"\n✅ Merged coordinate data for {df['latitude'].notna().sum():,} measurements")
else:
    # Add empty coordinate columns if no data available
    df['latitude'] = None
    df['longitude'] = None
    df['road_name'] = None
    df['direction'] = None
    print("\n⚠️  No coordinate data available to merge")

# Display basic information
print("\nDataFrame created successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:")
print(df.dtypes)

In [ ]:
# Display first few rows
print("\nFirst 10 rows:")
df.head(10)

## 6. Basic Statistics and Data Overview

In [ ]:
# Separate flow and speed data
df_flow = df[df['measurement_type'] == 'flow'].copy()
df_speed = df[df['measurement_type'] == 'speed'].copy()

print("=" * 60)
print("TRAFFIC DATA OVERVIEW")
print("=" * 60)

print(f"\n📊 Total measurements: {len(df):,}")
print(f"   - Flow measurements: {len(df_flow):,}")
print(f"   - Speed measurements: {len(df_speed):,}")

print(f"\n📍 Unique measurement sites: {df['site_id'].nunique()}")
print(f"\n⏰ Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")

In [ ]:
# Traffic Flow Statistics
print("\n" + "=" * 60)
print("TRAFFIC FLOW STATISTICS (vehicles/hour)")
print("=" * 60)

# Filter out zero flows for better statistics
df_flow_nonzero = df_flow[df_flow['value'] > 0]

print(f"\nNon-zero flow measurements: {len(df_flow_nonzero):,} ({len(df_flow_nonzero)/len(df_flow)*100:.1f}%)")
print(f"\nStatistics for non-zero flows:")
print(df_flow_nonzero['value'].describe())

In [ ]:
# Traffic Speed Statistics
print("\n" + "=" * 60)
print("TRAFFIC SPEED STATISTICS (km/h)")
print("=" * 60)

# Filter out invalid speeds (-1 indicates no measurement)
df_speed_valid = df_speed[df_speed['value'] > 0]

print(f"\nValid speed measurements: {len(df_speed_valid):,} ({len(df_speed_valid)/len(df_speed)*100:.1f}%)")
print(f"\nStatistics for valid speeds:")
print(df_speed_valid['value'].describe())

# Additional speed quality metrics
if 'std_deviation' in df_speed_valid.columns:
    print(f"\nAverage standard deviation: {df_speed_valid['std_deviation'].mean():.2f} km/h")
if 'num_values_used' in df_speed_valid.columns:
    print(f"Average number of values used: {df_speed_valid['num_values_used'].mean():.1f}")

## 6. Data Visualizations

In [ ]:
# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Traffic Data Distribution Analysis', fontsize=16, y=1.02)

# 1. Traffic Flow Distribution
ax1 = axes[0, 0]
flow_values = df_flow_nonzero['value']
ax1.hist(flow_values, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
ax1.axvline(flow_values.median(), color='red', linestyle='--', label=f'Median: {flow_values.median():.0f}')
ax1.set_xlabel('Flow Rate (vehicles/hour)')
ax1.set_ylabel('Frequency')
ax1.set_title('Traffic Flow Distribution (Non-zero values)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Traffic Speed Distribution
ax2 = axes[0, 1]
speed_values = df_speed_valid['value']
ax2.hist(speed_values, bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
ax2.axvline(speed_values.median(), color='red', linestyle='--', label=f'Median: {speed_values.median():.0f} km/h')
ax2.set_xlabel('Speed (km/h)')
ax2.set_ylabel('Frequency')
ax2.set_title('Traffic Speed Distribution (Valid measurements)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Measurement Sites by Type
ax3 = axes[1, 0]
site_counts = df.groupby(['site_id', 'measurement_type']).size().unstack(fill_value=0)
top_sites = site_counts.sum(axis=1).nlargest(15)
site_counts_top = site_counts.loc[top_sites.index]
site_counts_top.plot(kind='bar', stacked=True, ax=ax3, color=['skyblue', 'lightgreen'])
ax3.set_xlabel('Measurement Site ID')
ax3.set_ylabel('Number of Measurements')
ax3.set_title('Top 15 Sites by Measurement Count')
ax3.legend(title='Type')
ax3.tick_params(axis='x', rotation=45)

# 4. Speed vs Flow Correlation (for sites with both)
ax4 = axes[1, 1]
# Get sites with both speed and flow measurements
sites_with_both = df.groupby('site_id')['measurement_type'].nunique()
sites_with_both = sites_with_both[sites_with_both == 2].index

# Calculate average speed and flow for each site
site_stats = []
for site in sites_with_both[:50]:  # Limit to first 50 sites for clarity
    avg_flow = df_flow_nonzero[df_flow_nonzero['site_id'] == site]['value'].mean()
    avg_speed = df_speed_valid[df_speed_valid['site_id'] == site]['value'].mean()
    if pd.notna(avg_flow) and pd.notna(avg_speed):
        site_stats.append({'flow': avg_flow, 'speed': avg_speed})

if site_stats:
    site_stats_df = pd.DataFrame(site_stats)
    ax4.scatter(site_stats_df['flow'], site_stats_df['speed'], alpha=0.6, s=50)
    ax4.set_xlabel('Average Flow Rate (vehicles/hour)')
    ax4.set_ylabel('Average Speed (km/h)')
    ax4.set_title('Speed vs Flow Relationship by Site')
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Traffic Patterns Analysis

In [ ]:
# Analyze speed categories
def categorize_speed(speed):
    if speed < 0:
        return 'No Data'
    elif speed == 0:
        return 'Standstill'
    elif speed < 30:
        return 'Very Slow'
    elif speed < 50:
        return 'Slow'
    elif speed < 80:
        return 'Normal'
    elif speed < 100:
        return 'Fast'
    else:
        return 'Very Fast'

df_speed['speed_category'] = df_speed['value'].apply(categorize_speed)

# Count speed categories
speed_categories = df_speed['speed_category'].value_counts()

# Create pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Speed categories pie chart
colors = ['lightgray', 'darkred', 'red', 'orange', 'green', 'lightgreen', 'darkgreen']
ax1.pie(speed_categories.values, labels=speed_categories.index, autopct='%1.1f%%', 
        colors=colors[:len(speed_categories)], startangle=90)
ax1.set_title('Distribution of Speed Categories', fontsize=14)

# Flow categories
def categorize_flow(flow):
    if flow == 0:
        return 'No Traffic'
    elif flow < 500:
        return 'Light'
    elif flow < 1000:
        return 'Moderate'
    elif flow < 1500:
        return 'Heavy'
    else:
        return 'Very Heavy'

df_flow['flow_category'] = df_flow['value'].apply(categorize_flow)
flow_categories = df_flow['flow_category'].value_counts()

# Flow categories pie chart
colors_flow = ['lightgray', 'lightblue', 'blue', 'darkblue', 'navy']
ax2.pie(flow_categories.values, labels=flow_categories.index, autopct='%1.1f%%',
        colors=colors_flow[:len(flow_categories)], startangle=90)
ax2.set_title('Distribution of Traffic Flow Categories', fontsize=14)

plt.tight_layout()
plt.show()

## 8. Site-Level Analysis

In [ ]:
# Aggregate statistics by site
site_summary = []

for site_id in df['site_id'].unique():
    site_data = df[df['site_id'] == site_id]
    
    # Flow statistics
    flow_data = site_data[site_data['measurement_type'] == 'flow']['value']
    flow_nonzero = flow_data[flow_data > 0]
    
    # Speed statistics
    speed_data = site_data[site_data['measurement_type'] == 'speed']['value']
    speed_valid = speed_data[speed_data > 0]
    
    # Get coordinates from first measurement (same for all measurements at this site)
    first_measurement = site_data.iloc[0]
    
    site_summary.append({
        'site_id': site_id,
        'latitude': first_measurement.get('latitude'),
        'longitude': first_measurement.get('longitude'),
        'road_name': first_measurement.get('road_name'),
        'direction': first_measurement.get('direction'),
        'total_measurements': len(site_data),
        'flow_measurements': len(flow_data),
        'speed_measurements': len(speed_data),
        'avg_flow': flow_nonzero.mean() if len(flow_nonzero) > 0 else np.nan,
        'max_flow': flow_nonzero.max() if len(flow_nonzero) > 0 else np.nan,
        'avg_speed': speed_valid.mean() if len(speed_valid) > 0 else np.nan,
        'max_speed': speed_valid.max() if len(speed_valid) > 0 else np.nan,
        'min_speed': speed_valid.min() if len(speed_valid) > 0 else np.nan,
        'speed_std': speed_valid.std() if len(speed_valid) > 0 else np.nan
    })

df_site_summary = pd.DataFrame(site_summary)
df_site_summary = df_site_summary.sort_values('total_measurements', ascending=False)

print("Top 10 Sites by Total Measurements:")
print(df_site_summary.head(10)[['site_id', 'total_measurements', 'avg_flow', 'avg_speed']].to_string())

# Show coordinate availability
sites_with_coords = df_site_summary['latitude'].notna().sum()
print(f"\n📍 Sites with coordinate data: {sites_with_coords:,} / {len(df_site_summary):,} ({sites_with_coords/len(df_site_summary)*100:.1f}%)")

In [ ]:
# Visualize coordinate coverage
if df_site_summary['latitude'].notna().sum() > 0:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Map of measurement sites
    sites_with_coords = df_site_summary[df_site_summary['latitude'].notna()].copy()
    
    # Color by average speed
    scatter = axes[0].scatter(sites_with_coords['longitude'], 
                              sites_with_coords['latitude'],
                              c=sites_with_coords['avg_speed'],
                              cmap='RdYlGn',
                              alpha=0.6,
                              s=50,
                              edgecolors='black',
                              linewidth=0.5)
    axes[0].set_xlabel('Longitude')
    axes[0].set_ylabel('Latitude')
    axes[0].set_title('Traffic Measurement Sites (colored by avg speed)')
    axes[0].grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=axes[0], label='Avg Speed (km/h)')
    
    # Plot 2: Map colored by traffic flow
    scatter2 = axes[1].scatter(sites_with_coords['longitude'], 
                               sites_with_coords['latitude'],
                               c=sites_with_coords['avg_flow'],
                               cmap='YlOrRd',
                               alpha=0.6,
                               s=50,
                               edgecolors='black',
                               linewidth=0.5)
    axes[1].set_xlabel('Longitude')
    axes[1].set_ylabel('Latitude')
    axes[1].set_title('Traffic Measurement Sites (colored by avg flow)')
    axes[1].grid(True, alpha=0.3)
    plt.colorbar(scatter2, ax=axes[1], label='Avg Flow (veh/h)')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Visualized {len(sites_with_coords)} sites with coordinate data")
else:
    print("⚠️  No coordinate data available for visualization")

In [ ]:
# Identify interesting patterns
print("\n" + "=" * 60)
print("INTERESTING TRAFFIC PATTERNS")
print("=" * 60)

# Sites with highest average speed
print("\n🏎️ Top 5 Sites with Highest Average Speed:")
top_speed_sites = df_site_summary.nlargest(5, 'avg_speed')[['site_id', 'avg_speed']]
for _, row in top_speed_sites.iterrows():
    print(f"   {row['site_id']}: {row['avg_speed']:.1f} km/h")

# Sites with highest traffic flow
print("\n🚗 Top 5 Sites with Highest Average Flow:")
top_flow_sites = df_site_summary.nlargest(5, 'avg_flow')[['site_id', 'avg_flow']]
for _, row in top_flow_sites.iterrows():
    print(f"   {row['site_id']}: {row['avg_flow']:.0f} vehicles/hour")

# Sites with most variable speeds (high std)
print("\n📊 Top 5 Sites with Most Variable Speeds:")
variable_sites = df_site_summary.nlargest(5, 'speed_std')[['site_id', 'speed_std', 'avg_speed']]
for _, row in variable_sites.iterrows():
    print(f"   {row['site_id']}: σ = {row['speed_std']:.1f} km/h (avg: {row['avg_speed']:.1f} km/h)")

# Potential congestion sites (low speed, high flow)
print("\n🚦 Potential Congestion Sites (Low Speed, High Flow):")
congestion_candidates = df_site_summary[
    (df_site_summary['avg_speed'] < 40) & 
    (df_site_summary['avg_flow'] > 500)
][['site_id', 'avg_speed', 'avg_flow']]
if len(congestion_candidates) > 0:
    for _, row in congestion_candidates.head(5).iterrows():
        print(f"   {row['site_id']}: {row['avg_speed']:.1f} km/h, {row['avg_flow']:.0f} vehicles/hour")
else:
    print("   No significant congestion detected in this dataset")

## 9. Data Quality Analysis

In [ ]:
# Analyze data quality
print("=" * 60)
print("DATA QUALITY REPORT")
print("=" * 60)

# Missing or invalid data analysis
total_flow = len(df_flow)
zero_flow = len(df_flow[df_flow['value'] == 0])
nonzero_flow = len(df_flow[df_flow['value'] > 0])

total_speed = len(df_speed)
invalid_speed = len(df_speed[df_speed['value'] < 0])
valid_speed = len(df_speed[df_speed['value'] >= 0])

print(f"\n📊 Flow Measurements:")
print(f"   Total: {total_flow:,}")
print(f"   Zero flow: {zero_flow:,} ({zero_flow/total_flow*100:.1f}%)")
print(f"   Non-zero flow: {nonzero_flow:,} ({nonzero_flow/total_flow*100:.1f}%)")

print(f"\n🚗 Speed Measurements:")
print(f"   Total: {total_speed:,}")
print(f"   Invalid (< 0): {invalid_speed:,} ({invalid_speed/total_speed*100:.1f}%)")
print(f"   Valid: {valid_speed:,} ({valid_speed/total_speed*100:.1f}%)")

# Quality metrics for speed measurements
if 'num_values_used' in df_speed.columns:
    print(f"\n📈 Speed Measurement Quality:")
    speed_with_values = df_speed[df_speed['num_values_used'].notna()]
    if len(speed_with_values) > 0:
        print(f"   Average samples per measurement: {speed_with_values['num_values_used'].mean():.1f}")
        print(f"   Max samples: {speed_with_values['num_values_used'].max():.0f}")
        print(f"   Min samples (non-zero): {speed_with_values[speed_with_values['num_values_used'] > 0]['num_values_used'].min():.0f}")

# Outlier detection
print(f"\n⚠️ Potential Outliers:")
speed_outliers = df_speed_valid[df_speed_valid['value'] > 150]  # Speeds above 150 km/h
print(f"   Extremely high speeds (>150 km/h): {len(speed_outliers)} measurements")
if len(speed_outliers) > 0:
    print(f"   Sites with extreme speeds: {speed_outliers['site_id'].nunique()}")

flow_outliers = df_flow[df_flow['value'] > 3000]  # Very high flow rates
print(f"   Extremely high flows (>3000 veh/h): {len(flow_outliers)} measurements")
if len(flow_outliers) > 0:
    print(f"   Sites with extreme flows: {flow_outliers['site_id'].nunique()}")

## 11. Export Processed Data

In [ ]:
# Create cleaned datasets for further analysis
print("Creating cleaned datasets...")

# Clean flow data (keep coordinates)
df_flow_clean = df_flow[df_flow['value'] > 0].copy()
df_flow_clean = df_flow_clean.drop(columns=['measurement_type', 'unit'])
df_flow_clean.rename(columns={'value': 'flow_rate'}, inplace=True)

# Clean speed data (keep coordinates)
df_speed_clean = df_speed[df_speed['value'] > 0].copy()
df_speed_clean = df_speed_clean.drop(columns=['measurement_type', 'unit'])
df_speed_clean.rename(columns={'value': 'speed'}, inplace=True)

# Display columns being exported
print(f"\nFlow data columns: {df_flow_clean.columns.tolist()}")
print(f"Speed data columns: {df_speed_clean.columns.tolist()}")
print(f"Site summary columns: {df_site_summary.columns.tolist()}")

# Save to CSV files
output_dir = r'C:\Users\nicol\Documents\TrafficOntology_Project\TrafficOntology\data_processed\traffic_flow\'
flow_file = output_dir + 'traffic_flow_clean.csv'
speed_file = output_dir + 'traffic_speed_clean.csv'
site_summary_file = output_dir + 'site_summary.csv'

df_flow_clean.to_csv(flow_file, index=False)
df_speed_clean.to_csv(speed_file, index=False)
df_site_summary.to_csv(site_summary_file, index=False)

print(f"\n✅ Data exported successfully:")
print(f"   - Flow data: {flow_file} ({len(df_flow_clean):,} rows)")
print(f"   - Speed data: {speed_file} ({len(df_speed_clean):,} rows)")
print(f"   - Site summary: {site_summary_file} ({len(df_site_summary):,} sites)")
print(f"\n📍 All files include latitude, longitude, road_name, and direction columns (where available)")

## 12. Advanced Analysis - Traffic State Classification

In [ ]:
# Create traffic state classification based on flow and speed
def classify_traffic_state(row):
    """
    Classify traffic state based on flow and speed using fundamental traffic flow theory.
    """
    if pd.isna(row['avg_flow']) or pd.isna(row['avg_speed']):
        return 'Unknown'
    
    flow = row['avg_flow']
    speed = row['avg_speed']
    
    if speed > 80 and flow < 1000:
        return 'Free Flow'
    elif speed > 60 and flow >= 1000:
        return 'High Density Flow'
    elif 30 <= speed <= 60:
        return 'Synchronized Flow'
    elif speed < 30 and flow > 500:
        return 'Congested'
    elif speed < 30 and flow <= 500:
        return 'Jam'
    else:
        return 'Transitional'

df_site_summary['traffic_state'] = df_site_summary.apply(classify_traffic_state, axis=1)

# Visualize traffic states
traffic_states = df_site_summary['traffic_state'].value_counts()

plt.figure(figsize=(12, 6))

# Bar plot of traffic states
plt.subplot(1, 2, 1)
traffic_states.plot(kind='bar', color=['green', 'yellow', 'orange', 'red', 'darkred', 'gray', 'lightgray'])
plt.title('Distribution of Traffic States Across Sites')
plt.xlabel('Traffic State')
plt.ylabel('Number of Sites')
plt.xticks(rotation=45)

# Scatter plot of speed vs flow colored by state
plt.subplot(1, 2, 2)
colors = {'Free Flow': 'green', 'High Density Flow': 'yellow', 
          'Synchronized Flow': 'orange', 'Congested': 'red', 
          'Jam': 'darkred', 'Transitional': 'gray', 'Unknown': 'lightgray'}

for state in colors:
    state_data = df_site_summary[df_site_summary['traffic_state'] == state]
    plt.scatter(state_data['avg_flow'], state_data['avg_speed'], 
               label=state, color=colors[state], alpha=0.7, s=30)

plt.xlabel('Average Flow (vehicles/hour)')
plt.ylabel('Average Speed (km/h)')
plt.title('Traffic State Classification (Speed-Flow Diagram)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Traffic State Distribution:")
for state, count in traffic_states.items():
    print(f"  {state}: {count} sites ({count/len(df_site_summary)*100:.1f}%)")

## 13. Summary and Insights

In [ ]:
print("=" * 70)
print("TRAFFIC DATA ANALYSIS SUMMARY")
print("=" * 70)

print("\n📊 KEY FINDINGS:\n")

# Calculate key metrics
overall_avg_speed = df_speed_valid['value'].mean()
overall_median_speed = df_speed_valid['value'].median()
overall_avg_flow = df_flow_nonzero['value'].mean()
overall_median_flow = df_flow_nonzero['value'].median()

print(f"1. TRAFFIC SPEED:")
print(f"   • Average speed across all sites: {overall_avg_speed:.1f} km/h")
print(f"   • Median speed: {overall_median_speed:.1f} km/h")
print(f"   • Speed range: {df_speed_valid['value'].min():.0f} - {df_speed_valid['value'].max():.0f} km/h")
print(f"   • {(df_speed_valid['value'] < 50).sum()/len(df_speed_valid)*100:.1f}% of measurements show slow traffic (<50 km/h)")

print(f"\n2. TRAFFIC FLOW:")
print(f"   • Average flow rate: {overall_avg_flow:.0f} vehicles/hour")
print(f"   • Median flow rate: {overall_median_flow:.0f} vehicles/hour")
print(f"   • Maximum observed flow: {df_flow_nonzero['value'].max():.0f} vehicles/hour")
print(f"   • {(df_flow['value'] == 0).sum()/len(df_flow)*100:.1f}% of measurements show no traffic")

print(f"\n3. NETWORK COVERAGE:")
print(f"   • Total measurement sites: {df['site_id'].nunique()}")
print(f"   • Sites with both speed and flow data: {len(sites_with_both)}")
print(f"   • Average measurements per site: {len(df)/df['site_id'].nunique():.0f}")

print(f"\n4. DATA QUALITY:")
print(f"   • Valid speed measurements: {len(df_speed_valid)/len(df_speed)*100:.1f}%")
print(f"   • Non-zero flow measurements: {len(df_flow_nonzero)/len(df_flow)*100:.1f}%")
if 'std_deviation' in df_speed_valid.columns:
    avg_std = df_speed_valid['std_deviation'].mean()
    print(f"   • Average measurement uncertainty: ±{avg_std:.1f} km/h")

# Traffic conditions summary
print(f"\n5. TRAFFIC CONDITIONS:")
free_flow_pct = (df_site_summary['traffic_state'] == 'Free Flow').sum() / len(df_site_summary) * 100
congested_pct = df_site_summary['traffic_state'].isin(['Congested', 'Jam']).sum() / len(df_site_summary) * 100
print(f"   • Sites with free-flow conditions: {free_flow_pct:.1f}%")
print(f"   • Sites experiencing congestion: {congested_pct:.1f}%")

print("\n" + "=" * 70)
print("\n✅ Analysis complete! Processed data and visualizations are ready for further use.")
print("\n📁 Output files created:")
print("   • traffic_flow_clean.csv")
print("   • traffic_speed_clean.csv")
print("   • site_summary.csv")